In [ ]:
# -*- coding: utf-8 -*-

When Accuracy Lies: A Multi-Dimensional ML Benchmark.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1j720LeSH7EaIch4ZfXVeIsdUNP6uXeDa

## Which Machine-Learning Models Fail Gracefully?
## Creating An Optimized Version Multi-Dimensional ML Benchmark

Machine-learning models are commonly compared using performance on a clean holdout set. In real deployment environments, however, data quality rarely remains ideal. Features may become noisy, values may go missing, extreme observations may appear, labels may be wrong, and the statistical distribution of incoming data can shift.

A model that achieves the highest clean-test score may therefore not be the model that behaves most reliably when conditions deteriorate.

> **Core research question:** How stable are different machine-learning models when the assumptions of clean training and test data are progressively violated?

Rather than asking only **“Which model performs best?”**, this notebook asks:

> **“Which models degrade gradually, and which models collapse?”**

## Research Card

| Component | Design |
|---|---|
| Datasets | 3 tabular classification datasets |
| Models | 6 representative model families |
| Stress Tests | 6 |
| Random Seeds | 2 in Fast Mode, 5 in Full Mode |
| Primary Metric | Weighted F1 |
| Secondary Metrics | Accuracy, Precision, Recall, Balanced Accuracy, ROC-AUC |
| Robustness Metrics | Relative Degradation, Stability Score, Degradation Slope, GFI |
| Goal | Evaluate reliability beyond clean-test performance |

### Why this version is optimized

The earlier implementation retrained models at every corruption level. That is unnecessary for stress tests that alter only the test data.

This version trains each model **once per dataset and seed** and reuses it for:

- Feature Noise
- Missing Data
- Outliers
- Distribution Shift
- Feature Corruption

Only **Label Noise** triggers retraining because it changes the training target.

# 1. Research Question and Hypotheses

## Research Question

> **Which model families maintain predictive performance most consistently as data quality and distribution deteriorate?**

The notebook does not attempt to identify a universal winner. It measures **degradation profiles**.

### H1
Models will exhibit different degradation patterns under the same corruption.

### H2
The model with the highest clean-test performance will not necessarily show the lowest degradation.

### H3
Different model families will show different robustness profiles depending on corruption type.

### H4
Model rankings observed on clean data may change under severe corruption or distribution shift.

# 2. Experimental Protocol

For each dataset and random seed:

```text
Dataset
   ↓
Train/Test Split
   ↓
Preprocessing Pipeline
   ↓
Train Model
   ↓
Clean Baseline
   ↓
Controlled Corruption
   ↓
Repeated Evaluation
   ↓
Degradation Analysis
   ↓
Stability Analysis
```

Five experiments represent **test-time corruption** and reuse the fitted model. Label noise represents **training-time corruption** and requires retraining.

## 3. Imports and Runtime Configuration

In [ ]:

import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_wine, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, balanced_accuracy_score

warnings.filterwarnings("ignore")

FAST_MODE = True

if FAST_MODE:
    RANDOM_SEEDS = [42, 52]
    RF_TREES = 100
    MLP_MAX_ITER = 250
else:
    RANDOM_SEEDS = [42, 52, 62, 72, 82]
    RF_TREES = 250
    MLP_MAX_ITER = 500

PRIMARY_METRIC = "f1"
print("FAST_MODE:", FAST_MODE)
print("Seeds:", RANDOM_SEEDS)

### Runtime Modes

**Fast Mode** is intended for laptops and normal Kaggle use. It uses fewer seeds, fewer Random Forest trees, fewer MLP iterations, and fewer corruption levels.

**Full Mode** increases experimental depth for a final benchmark.

Use:

```python
FAST_MODE = True
```

for development and normal execution, then switch to:

```python
FAST_MODE = False
```

for the final extended run.

# 4. Dataset Strategy

A conclusion based on one dataset can easily be dataset-specific. This benchmark therefore uses three datasets with different structures.

### Breast Cancer Wisconsin
Small binary classification.

### Wine
Multiclass classification with continuous chemical measurements.

### Digits
Higher-dimensional multiclass classification using pixel-derived features.

This provides variation in sample size, dimensionality, number of classes, and feature distributions.

In [ ]:

def load_benchmark_datasets(include_digits=True):
    datasets = {}

    cancer = load_breast_cancer()
    datasets["Breast Cancer"] = {
        "X": pd.DataFrame(cancer.data, columns=cancer.feature_names),
        "y": pd.Series(cancer.target)
    }

    wine = load_wine()
    datasets["Wine"] = {
        "X": pd.DataFrame(wine.data, columns=wine.feature_names),
        "y": pd.Series(wine.target)
    }

    if include_digits:
        digits = load_digits()
        cols = [f"pixel_{i}" for i in range(digits.data.shape[1])]
        datasets["Digits"] = {
            "X": pd.DataFrame(digits.data, columns=cols),
            "y": pd.Series(digits.target)
        }

    return datasets

datasets = load_benchmark_datasets(include_digits=True)

pd.DataFrame([{
    "Dataset": name,
    "Rows": obj["X"].shape[0],
    "Features": obj["X"].shape[1],
    "Classes": obj["y"].nunique()
} for name, obj in datasets.items()])

# 5. Models

The benchmark compares representative model families rather than a large collection of similar algorithms.

### Linear
- Logistic Regression

### Tree-based
- Decision Tree
- Random Forest

### Margin-based
- Support Vector Machine

### Boosting
- Histogram Gradient Boosting

### Neural
- Multi-Layer Perceptron

The model implementations are also tuned for practical runtime. SVM probability calibration is disabled, Random Forest uses fewer trees in Fast Mode, Histogram Gradient Boosting replaces the slower classic implementation, and the MLP uses early stopping.

In [ ]:

def build_models(seed=42):
    return {
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=seed))
        ]),
        "Decision Tree": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(random_state=seed))
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(n_estimators=RF_TREES, random_state=seed, n_jobs=-1))
        ]),
        "SVM": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(probability=False, random_state=seed))
        ]),
        "Hist Gradient Boosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingClassifier(max_iter=100 if FAST_MODE else 200, random_state=seed))
        ]),
        "MLP": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(48,24) if FAST_MODE else (64,32),
                max_iter=MLP_MAX_ITER,
                early_stopping=True,
                n_iter_no_change=12,
                random_state=seed
            ))
        ])
    }

list(build_models().keys())

# 6. Evaluation Metrics

Accuracy alone is not sufficient for this benchmark.

### Primary metric
**Weighted F1 Score**

### Secondary metrics
- Accuracy
- Precision
- Recall
- Balanced Accuracy
- ROC-AUC

Weighted averaging makes the evaluation usable for both binary and multiclass datasets while accounting for class support.

In [ ]:

def evaluate_model(model, X, y):
    pred = model.predict(X)
    result = {
        "accuracy": accuracy_score(y, pred),
        "f1": f1_score(y, pred, average="weighted", zero_division=0),
        "precision": precision_score(y, pred, average="weighted", zero_division=0),
        "recall": recall_score(y, pred, average="weighted", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "roc_auc": np.nan
    }

    classes = np.unique(y)
    try:
        if hasattr(model, "predict_proba"):
            scores = model.predict_proba(X)
            if len(classes) == 2:
                result["roc_auc"] = roc_auc_score(y, scores[:,1])
            else:
                result["roc_auc"] = roc_auc_score(y, scores, multi_class="ovr", average="weighted")
        elif hasattr(model, "decision_function") and len(classes) == 2:
            scores = model.decision_function(X)
            result["roc_auc"] = roc_auc_score(y, scores)
    except Exception:
        pass

    return result

# 7. Stress Tests

## Experiment A: Feature Noise
Gaussian noise is added to numerical feature values.

**Question:** How sensitive is each model to measurement noise?

## Experiment B: Missing Data
A proportion of feature values is randomly removed.

**Question:** How does performance change when information becomes incomplete?

## Experiment C: Outliers
A fraction of observations receives large numerical perturbations.

**Question:** Which models are sensitive to extreme observations?

## Experiment D: Label Noise
A proportion of training labels is replaced with incorrect labels.

**Question:** How sensitive is learning to incorrect supervision?

## Experiment E: Distribution Shift
Feature values are systematically shifted away from the training distribution.

**Question:** How well does the model generalize when deployment conditions differ?

## Experiment F: Feature Corruption
Selected feature columns are shuffled across observations.

**Question:** How dependent is a model on stable feature relationships?

In [ ]:

def add_gaussian_noise(X, level, random_state=42):
    if level == 0: return X.copy()
    rng = np.random.default_rng(random_state)
    Xc = X.copy()
    stds = Xc.std(axis=0).replace(0,1).to_numpy()
    noise = rng.normal(0, level, size=Xc.shape) * stds
    Xc.iloc[:,:] = Xc.to_numpy() + noise
    return Xc

def introduce_missingness(X, level, random_state=42):
    if level == 0: return X.copy()
    rng = np.random.default_rng(random_state)
    return X.mask(rng.random(X.shape) < level)

def inject_outliers(X, level, magnitude=5.0, random_state=42):
    if level == 0: return X.copy()
    rng = np.random.default_rng(random_state)
    Xc = X.copy()
    n_corrupt = max(1, int(len(Xc) * level))
    rows = rng.choice(len(Xc), size=n_corrupt, replace=False)
    stds = Xc.std(axis=0).replace(0,1).to_numpy()
    perturb = rng.normal(0, magnitude, size=(n_corrupt, Xc.shape[1])) * stds
    Xc.iloc[rows,:] = Xc.iloc[rows,:].to_numpy() + perturb
    return Xc

def apply_distribution_shift(X, level, random_state=42):
    if level == 0: return X.copy()
    return X + level * X.std(axis=0).replace(0,1)

def corrupt_features(X, level, random_state=42):
    if level == 0: return X.copy()
    rng = np.random.default_rng(random_state)
    Xc = X.copy()
    n = max(1, int(round(Xc.shape[1] * level)))
    cols = rng.choice(np.arange(Xc.shape[1]), size=min(n, Xc.shape[1]), replace=False)
    for j in cols:
        Xc.iloc[:,j] = rng.permutation(Xc.iloc[:,j].to_numpy())
    return Xc

def add_label_noise(y, level, random_state=42):
    if level == 0: return np.asarray(y).copy()
    rng = np.random.default_rng(random_state)
    yc = np.asarray(y).copy()
    n_flip = max(1, int(len(yc) * level))
    idx = rng.choice(len(yc), size=n_flip, replace=False)
    classes = np.unique(yc)
    for i in idx:
        yc[i] = rng.choice(classes[classes != yc[i]])
    return yc

# 8. Corruption Levels

Fast Mode uses a reduced but still informative set of severity levels. The objective is to preserve the shape of the degradation curve without creating unnecessary runtime.

Full Mode uses a denser set of corruption levels.

In [ ]:

if FAST_MODE:
    TEST_STRESS_CONFIG = {
        "Feature Noise": ([0.00,0.10,0.25,0.40], add_gaussian_noise),
        "Missing Data": ([0.00,0.10,0.25,0.40], introduce_missingness),
        "Outliers": ([0.00,0.05,0.10,0.20], inject_outliers),
        "Distribution Shift": ([0.00,0.50,1.00,2.00], apply_distribution_shift),
        "Feature Corruption": ([0.00,0.25,0.50,0.75], corrupt_features)
    }
    LABEL_NOISE_LEVELS = [0.00,0.10,0.20,0.30]
else:
    TEST_STRESS_CONFIG = {
        "Feature Noise": ([0.00,0.05,0.10,0.20,0.30,0.40], add_gaussian_noise),
        "Missing Data": ([0.00,0.05,0.10,0.20,0.30,0.40], introduce_missingness),
        "Outliers": ([0.00,0.01,0.05,0.10,0.20], inject_outliers),
        "Distribution Shift": ([0.00,0.25,0.50,1.00,2.00], apply_distribution_shift),
        "Feature Corruption": ([0.00,0.10,0.25,0.50,0.75], corrupt_features)
    }
    LABEL_NOISE_LEVELS = [0.00,0.05,0.10,0.20,0.30]

# 9. Optimized Benchmark Engine

A naive benchmark repeatedly performs:

```text
Dataset × Seed × Stress Test × Level × Model × Fit
```

The optimized version performs:

```text
Dataset × Seed × Model × One Clean Fit
```

and then reuses the fitted model across all **test-time** corruptions.

Only label noise causes retraining because the training labels themselves change.

This is the main computational improvement in the notebook.

In [ ]:

def run_optimized_benchmark(datasets, seeds=RANDOM_SEEDS, test_size=0.25):
    records = []
    fit_counter = 0
    start = time.time()

    for dataset_name, obj in datasets.items():
        X, y = obj["X"], obj["y"]
        print(f"\n=== {dataset_name} ===")

        for seed in seeds:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=test_size, stratify=y, random_state=seed
            )

            fitted = {}
            for model_name, model in build_models(seed).items():
                t0 = time.time()
                model.fit(X_train, y_train)
                fit_time = time.time() - t0
                fit_counter += 1
                fitted[model_name] = model

                metrics = evaluate_model(model, X_test, y_test)
                records.append({
                    "dataset": dataset_name, "seed": seed, "model": model_name,
                    "corruption_type": "Clean", "corruption_level": 0.0,
                    **metrics, "fit_time_sec": fit_time
                })

            for stress_name, (levels, func) in TEST_STRESS_CONFIG.items():
                for level in levels:
                    X_eval = X_test if level == 0 else func(
                        X_test, level, random_state=seed + int(level*1000)
                    )
                    for model_name, model in fitted.items():
                        metrics = evaluate_model(model, X_eval, y_test)
                        records.append({
                            "dataset": dataset_name, "seed": seed, "model": model_name,
                            "corruption_type": stress_name, "corruption_level": level,
                            **metrics, "fit_time_sec": 0.0
                        })

            for level in LABEL_NOISE_LEVELS:
                if level == 0:
                    for model_name, model in fitted.items():
                        metrics = evaluate_model(model, X_test, y_test)
                        records.append({
                            "dataset": dataset_name, "seed": seed, "model": model_name,
                            "corruption_type": "Label Noise", "corruption_level": 0.0,
                            **metrics, "fit_time_sec": 0.0
                        })
                else:
                    y_noisy = add_label_noise(y_train, level, random_state=seed + int(level*1000))
                    for model_name, model in build_models(seed).items():
                        t0 = time.time()
                        model.fit(X_train, y_noisy)
                        fit_time = time.time() - t0
                        fit_counter += 1
                        metrics = evaluate_model(model, X_test, y_test)
                        records.append({
                            "dataset": dataset_name, "seed": seed, "model": model_name,
                            "corruption_type": "Label Noise", "corruption_level": level,
                            **metrics, "fit_time_sec": fit_time
                        })

            print(f"Seed {seed} complete | fits so far: {fit_counter}")

    elapsed = time.time() - start
    print(f"\nTotal model fits: {fit_counter}")
    print(f"Elapsed time: {elapsed/60:.2f} minutes")
    return pd.DataFrame(records)

## Estimated Fit Reduction

The next cell compares the approximate number of model fits under the repeated-training design with the optimized design.

Prediction still has a cost, but model fitting is usually the dominant part of this benchmark.

In [ ]:

n_datasets = len(datasets)
n_models = len(build_models())
n_seeds = len(RANDOM_SEEDS)
old_levels = sum(len(v[0]) for v in TEST_STRESS_CONFIG.values()) + len(LABEL_NOISE_LEVELS)
old_fit_estimate = n_datasets * n_seeds * n_models * old_levels
new_fit_estimate = n_datasets * n_seeds * n_models * (1 + len(LABEL_NOISE_LEVELS)-1)
print("Repeated-training design fits:", old_fit_estimate)
print("Optimized design fits:", new_fit_estimate)
print("Fit reduction: %.1f%%" % (100*(1-new_fit_estimate/old_fit_estimate)))

# 10. Run the Benchmark

The benchmark now executes the full workflow:

1. Split each dataset.
2. Fit clean models once.
3. Record clean performance.
4. Generate corrupted test sets.
5. Reuse fitted models across test-time stress tests.
6. Retrain only for label noise.
7. Store all performance metrics and runtime information.

For a very quick development run, you can temporarily use one seed and exclude Digits.

In [ ]:

benchmark_results = run_optimized_benchmark(
    datasets=datasets,
    seeds=RANDOM_SEEDS,
    test_size=0.25
)
benchmark_results.head()

# 11. Relative Degradation and Stability

Raw scores alone do not show how much performance a model loses relative to its own clean baseline.

Relative degradation is:

\[
D = \frac{S_{clean}-S_{corrupted}}{S_{clean}}
\]

For example, if clean F1 is 0.95 and corrupted F1 is 0.85:

\[
D = \frac{0.95-0.85}{0.95} \approx 0.105
\]

So the model loses about **10.5%** of its clean performance.

Normalized relative performance is:

\[
R_i=\frac{Score_i}{Score_{clean}}
\]

and the Stability Score is the mean of these normalized scores across corruption levels.

A score near **1.0** indicates that the model retains most of its clean performance.

In [ ]:

def add_relative_degradation(df, metric="f1"):
    stress = df[df["corruption_type"] != "Clean"].copy()
    baseline = (
        stress[stress["corruption_level"] == 0]
        .set_index(["dataset","seed","model","corruption_type"])[metric]
        .to_dict()
    )
    stress[f"baseline_{metric}"] = stress.apply(
        lambda r: baseline[(r["dataset"],r["seed"],r["model"],r["corruption_type"])], axis=1
    )
    denom = stress[f"baseline_{metric}"].replace(0,np.nan)
    stress["relative_degradation"] = (stress[f"baseline_{metric}"] - stress[metric]) / denom
    stress["relative_performance"] = stress[metric] / denom
    return stress

stress_results = add_relative_degradation(benchmark_results)

stability_summary = (
    stress_results.groupby(["dataset","model","corruption_type"], as_index=False)
    .agg(stability_score=("relative_performance","mean"),
         mean_degradation=("relative_degradation","mean"))
)
stability_summary.head()

# 12. Degradation Slope

A simple regression line is fitted to each degradation curve:

\[
Performance = \beta_0 + \beta_1(Corruption)
\]

The slope \(\beta_1\) provides a compact description of how quickly performance changes.

- Near zero: relatively stable
- Moderately negative: gradual degradation
- Strongly negative: rapid degradation

The relationship need not be perfectly linear. The slope is used as a descriptive summary.

In [ ]:

def compute_slopes(df, metric="f1"):
    rows=[]
    for keys, sub in df.groupby(["dataset","seed","model","corruption_type"]):
        reg = LinearRegression().fit(sub[["corruption_level"]], sub[metric])
        rows.append({
            "dataset":keys[0], "seed":keys[1], "model":keys[2],
            "corruption_type":keys[3], "degradation_slope":reg.coef_[0]
        })
    return pd.DataFrame(rows)

slope_results = compute_slopes(stress_results)
slope_results.head()

# 13. Graceful Failure Index

The notebook also calculates a GFI-style measure based on the normalized area under the relative-performance curve.

A model that maintains performance across the full corruption range will have a larger normalized area.

### Important
The GFI used here is a **proposed benchmark summary metric**, not an established universal machine-learning metric.

In [ ]:

def compute_gfi(df):
    rows=[]
    for keys, sub in df.groupby(["dataset","seed","model","corruption_type"]):
        sub = sub.sort_values("corruption_level")
        x = sub["corruption_level"].to_numpy(float)
        y = sub["relative_performance"].to_numpy(float)
        gfi = np.nan if len(x)<2 or x.max()==x.min() else np.trapz(y,x)/(x.max()-x.min())
        rows.append({
            "dataset":keys[0], "seed":keys[1], "model":keys[2],
            "corruption_type":keys[3], "gfi":gfi
        })
    return pd.DataFrame(rows)

gfi_results = compute_gfi(stress_results)
gfi_results.head()

# 14. Performance Degradation Curves

The degradation curve is one of the central visualizations.

The starting point shows clean performance, while the shape and steepness of the curve reveal how gradually or abruptly a model deteriorates as corruption increases.

In [ ]:

def plot_degradation_curves(df, dataset_name, corruption_type, metric="f1"):
    sub = df[(df["dataset"]==dataset_name) & (df["corruption_type"]==corruption_type)]
    summary = sub.groupby(["model","corruption_level"], as_index=False)[metric].mean()
    plt.figure(figsize=(10,6))
    for name in summary["model"].unique():
        m = summary[summary["model"]==name]
        plt.plot(m["corruption_level"], m[metric], marker="o", label=name)
    plt.xlabel("Corruption Level")
    plt.ylabel(metric.upper())
    plt.title(f"{dataset_name}: {corruption_type}")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

plot_degradation_curves(stress_results, "Breast Cancer", "Feature Noise")

# 15. Stability Heatmap

The heatmap summarizes robustness across multiple stress tests.

Rows represent models and columns represent corruption types. This makes it easier to see whether a model is broadly stable or robust only to specific forms of corruption.

In [ ]:

def plot_stability_heatmap(stability_df, dataset_name):
    pivot = stability_df[stability_df["dataset"]==dataset_name].pivot(
        index="model", columns="corruption_type", values="stability_score"
    )
    fig, ax = plt.subplots(figsize=(11,6))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j,i,f"{pivot.iloc[i,j]:.3f}",ha="center",va="center")
    ax.set_title(f"{dataset_name}: Stability Score")
    fig.colorbar(im, ax=ax, label="Stability Score")
    plt.tight_layout()
    plt.show()

for dataset_name in datasets:
    plot_stability_heatmap(stability_summary, dataset_name)

# 16. Clean Performance vs Robustness

A central idea in this notebook is that clean predictive strength and robustness are not necessarily the same thing.

The scatter plot compares:

- **x-axis:** clean F1
- **y-axis:** overall Stability Score

This directly tests whether the strongest clean model is also the model that best preserves its performance.

In [ ]:

clean_summary = (
    benchmark_results[benchmark_results["corruption_type"]=="Clean"]
    .groupby(["dataset","model"], as_index=False)
    .agg(clean_f1=("f1","mean"), clean_accuracy=("accuracy","mean"))
)

overall_stability = (
    stability_summary.groupby(["dataset","model"], as_index=False)
    .agg(overall_stability=("stability_score","mean"))
)

clean_vs_stability = clean_summary.merge(overall_stability,on=["dataset","model"])
clean_vs_stability

# 17. Final Multi-Dimensional Summary

The final summary combines:

- Clean F1
- Clean Accuracy
- Mean Stability
- Mean Relative Degradation
- Mean GFI
- Mean Degradation Slope

The table should not be treated as a universal leaderboard. Its purpose is to show that different definitions of model quality can lead to different conclusions.

In [ ]:

final_summary = (
    clean_summary
    .merge(
        stability_summary.groupby(["dataset","model"], as_index=False)
        .agg(mean_stability=("stability_score","mean"), mean_relative_degradation=("mean_degradation","mean")),
        on=["dataset","model"]
    )
    .merge(
        gfi_results.groupby(["dataset","model"], as_index=False).agg(mean_gfi=("gfi","mean")),
        on=["dataset","model"]
    )
    .merge(
        slope_results.groupby(["dataset","model"], as_index=False).agg(mean_degradation_slope=("degradation_slope","mean")),
        on=["dataset","model"]
    )
)

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
benchmark_results.to_csv(os.path.join(OUTPUT_DIR,"optimized_benchmark_results.csv"),index=False)
stress_results.to_csv(os.path.join(OUTPUT_DIR,"optimized_stress_results.csv"),index=False)
stability_summary.to_csv(os.path.join(OUTPUT_DIR,"optimized_stability_summary.csv"),index=False)
final_summary.to_csv(os.path.join(OUTPUT_DIR,"optimized_final_summary.csv"),index=False)

final_summary

# 18. How to Interpret the Results

Do not end the notebook with:

> **“Model X is the best model.”**

Instead, interpret the results along several dimensions.

### Clean performance
Which models begin with the strongest F1?

### Relative degradation
How much does each model lose relative to its own baseline?

### Stability
Which models preserve performance most consistently?

### Degradation slope
Which models deteriorate rapidly?

### GFI
Which models preserve performance across the whole corruption range?

### Rank sensitivity
Would the model ordering change under severe corruption?

### Cross-dataset consistency
Does the same model behave similarly across datasets?

The important question is not simply:

> **Which model wins?**

It is:

> **How does model behavior change when the world stops looking like the training data?**

# 19. Suggested Findings Structure

After running the notebook, structure the discussion around the actual results.

Possible findings to test include:

1. Clean-test performance does not always predict robustness.
2. Different corruption mechanisms produce different degradation profiles.
3. Some models may remain stable under one stress test but degrade sharply under another.
4. Model ordering may change as corruption becomes severe.
5. Robustness may vary across datasets.
6. A single clean-test metric may therefore provide an incomplete picture of model reliability.

Retain only the statements that are supported by the empirical output.

# 20. Failure Analysis

Aggregate metrics are useful, but individual failure cases can be more informative.

Look for examples such as:

- high clean F1 with steep degradation,
- moderate clean F1 with strong stability,
- robustness to missing data but weakness under distribution shift,
- sharp performance loss after a particular corruption threshold.

Possible explanations can involve feature scaling, dimensionality, model complexity, regularization, nonlinear dependence, and sensitivity to specific features.

These explanations should be presented as interpretations unless additional analysis establishes causality.

# 21. Limitations

This benchmark has several limitations.

1. The datasets are relatively small and tabular.
2. Stress tests are controlled simulations rather than naturally occurring production failures.
3. Models use default or lightly configured hyperparameters.
4. Stability Score and GFI are proposed summary measures rather than universal standards.
5. The benchmark does not yet include calibration, fairness, adversarial perturbation, or feature-importance stability.
6. A larger study could use repeated cross-validation and more real-world datasets.

# 22. Possible Extensions

### Additional models
- XGBoost
- LightGBM
- K-Nearest Neighbors

### Additional stress tests
- Class imbalance
- Feature dropout
- Temporal drift
- Selected-feature covariate shift
- Adversarial perturbation

### Additional reliability measures
- Brier Score
- Expected Calibration Error
- Prediction confidence
- Feature-importance stability

### Stronger statistical design
- 10 random seeds
- repeated stratified cross-validation
- confidence intervals
- Friedman test
- post-hoc comparisons

A natural follow-up notebook would be:

> **Can We Trust Feature Importance? An Explainability Stability Benchmark**

# 23. Conclusion

Traditional model comparison usually asks:

> **Which model achieves the highest score on clean test data?**

This benchmark broadens the evaluation.

A reliable model should not only perform well when data is clean. It should also behave predictably when measurements become noisy, values disappear, outliers occur, labels contain errors, important features are disrupted, or deployment distributions change.

The core idea is therefore:

> **Accuracy describes how well a model performs under a particular test condition. Robustness describes how its behavior changes when those conditions stop being ideal.**

A complete evaluation should consider both.

### Benchmarking Philosophy

This notebook follows the broader principle of evaluating machine-learning algorithms under standardized and reproducible experimental conditions rather than relying on a single dataset or isolated model result.

Thank You!
If you found this notebook useful:

Upvote it

Leave a comment

Fork it for your own research paper metadata project

Follow my Kaggle profile

> Open to collaborations and suggestions.

Happy coding and happy learning!